Connect to Big Query

In [ ]:
from google.cloud import bigquery
from google.colab import auth

#authentication
project_id='your-gcp-project-id'
auth.authenticate_user(project_id=project_id)

#initialize the client for Big query
client=bigquery.Client(project=project_id, location='your-region')

In [ ]:
!pip install google-cloud-bigquery

Get the Dataset

In [ ]:
# getting the dataset
dataset_ref = client.dataset('employeesdata' , project=project_id)
dataset = client.get_dataset(dataset_ref)
table_ref = dataset.table('tbl-hr-data')
table = client.get_table(table_ref)
table.schema

In [ ]:
new_table_ref = dataset.table('tbl-new-employees')
new_table = client.get_table(new_table_ref)
new_table.schema

In [ ]:
#convert to dataframe

df= client.list_rows(table=table).to_dataframe()
df.head()

In [ ]:
#convert to dataframe

df2= client.list_rows(table=new_table).to_dataframe()
df2.head()

**Model** **Building**




# System Setup

In [ ]:
import sys
print(sys.version)

In [ ]:
!pip install -q "git+https://github.com/sktime/pycaret.git@v3.5.0"

In [ ]:
import pycaret
print(pycaret.__version__)

from pycaret.datasets import get_data
data = get_data('juice')

# Code and Train Model

In [ ]:
from pycaret.classification import*

In [ ]:
df.info()


In [ ]:
df.columns

In [ ]:
#setup or model
setup(df,target='Quit_the_Company',session_id=123,ignore_features=['employee_id'],categorical_features=['salary','Departments'])

In [ ]:
compare_models()

In [ ]:
rf_model=create_model('rf')

In [ ]:
final_df = predict_model(rf_model)
final_df.head

In [ ]:
final_df.head()

In [ ]:
final_df.info()

In [ ]:
new_predictions = predict_model(rf_model, data=df2)
new_predictions.head()

In [ ]:
new_predictions.to_gbq(
    'employeedata.pilot_predictions',
    project_id='your-gcp-project-id',
    chunksize=None,
    if_exists='replace'
)

In [ ]:
plot_model(rf_model,plot='feature')

In [ ]:
#create a feature table
rf_model.feature_names_in_

In [ ]:
rf_model.feature_importances_

In [ ]:
import pandas as pd
feature_table = pd.DataFrame(
    zip(rf_model.feature_names_in_, rf_model.feature_importances_),
    columns=['feature', 'importance']
)
feature_table.sort_values('importance', ascending=False)

In [ ]:
feature_table.to_gbq(
    'employeedata.feature_table',
    project_id='your-gcp-project-id',
    chunksize=None,
    if_exists='replace'
)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project-id')

tables_to_copy = ['tbl-full-data', 'tbl-hr-data', 'tbl-new-employees']

for table_name in tables_to_copy:
    source_table_id = f"your-gcp-project-id.employeesdata.{table_name}"
    destination_table_id = f"your-gcp-project-id.employeedata.{table_name}"

    try:
        # Get information about the source resource to check its type
        source_resource = client.get_table(source_table_id)

        if source_resource.table_type == "VIEW":
            print(f"Detected {source_table_id} as a VIEW. Copying its data to a new table.")

            # Configure the query job to write results to the destination table
            job_config = bigquery.QueryJobConfig(
                destination=destination_table_id,
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE # Overwrite if table exists
            )
            query = f"SELECT * FROM `{source_table_id}`"
            query_job = client.query(query, job_config=job_config)
            query_job.result() # Waits for the query job to complete
            print(f"Copied data from view {source_table_id} -> {destination_table_id}")
        else:
            # If it's a regular table, use copy_table
            copy_job = client.copy_table(source_table_id, destination_table_id)
            copy_job.result()  # waits for the copy to complete
            print(f"Copied table {source_table_id} -> {destination_table_id}")

    except Exception as e:
        print(f"Error processing {source_table_id}: {e}")

In [ ]:
query = """
SELECT table_name, table_type
FROM `your-gcp-project-id.employeedata.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""
client.query(query).to_dataframe()